In [1]:
import pandas as pd
import numpy as np

# -------------------------------
# Load CSV
# -------------------------------
base_path = "../../../"
goto_folder = "ResultGroup/2.Entanglement/"
filename = "Entanglement-Baseline1-Qwen3VL.csv"
df = pd.read_csv(f"{base_path}{goto_folder}{filename}")

# -------------------------------
# Task 1: Yes/No accuracy
# -------------------------------
def normalize_yesno(x):
    if isinstance(x, str):
        x = x.strip().lower()
        if x in ["yes", "no"]:
            return x
    return None

df["pred1_norm"] = df["prediction_1"].apply(normalize_yesno)
df["out1_norm"]  = df["output_1"].apply(normalize_yesno)

df["task1_correct"] = df["pred1_norm"] == df["out1_norm"]
task1_acc = df["task1_correct"].mean()

# -------------------------------
# Task 2: RMSE for purity
# -------------------------------
def to_float_safe(x):
    try:
        return float(x)
    except:
        return np.nan

df["pred2_float"] = df["prediction_2"].apply(to_float_safe)
df["out2_float"]  = df["output_2"].apply(to_float_safe)

# Ignore rows where ground truth is missing
mask_valid = ~df["out2_float"].isna()

# RMSE calculation
mse = ((df.loc[mask_valid, "pred2_float"] - df.loc[mask_valid, "out2_float"]) ** 2).mean()
rmse = np.sqrt(mse)

# -------------------------------
# Print summary
# -------------------------------
print("==== Evaluation Summary ====")
print(f"Yes/No Accuracy       : {task1_acc:.4f}")
print(f"Purity RMSE           : {rmse:.6f}")
print("Saved to quantum_vqa_eval_results.csv")


==== Evaluation Summary ====
Yes/No Accuracy       : 0.6892
Purity RMSE           : 0.311194
Saved to quantum_vqa_eval_results.csv


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, r2_score, mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr
import numpy as np
import os

# -------------------------------
# Load CSV
# -------------------------------
base_path = "../../../"
goto_folder = "ResultGroup/2.Entanglement/"
filename = "Entanglement-Baseline1-Qwen3VL.csv"
df = pd.read_csv(f"{base_path}{goto_folder}{filename}")


# --- Task 2: Purity Regression (prediction_2 vs output_2) ---
print("\n--- Task 2: Purity Estimation ---")
if 'prediction_2' in df.columns and 'output_2' in df.columns:
    y_pred_reg = pd.to_numeric(df['prediction_2'], errors='coerce')
    y_true_reg = pd.to_numeric(df['output_2'], errors='coerce')
    mask = ~np.isnan(y_pred_reg) & ~np.isnan(y_true_reg)
    y_pred_clean = y_pred_reg[mask]
    y_true_clean = y_true_reg[mask]
    
    if len(y_true_clean) > 0:
        # Metrics
        r2 = r2_score(y_true_clean, y_pred_clean)
        mae = mean_absolute_error(y_true_clean, y_pred_clean)
        pearson_corr, _ = pearsonr(y_true_clean, y_pred_clean)
        spearman_corr, _ = spearmanr(y_true_clean, y_pred_clean)
        
        print(f"R2 Score: {r2:.4f}")
        print(f"MAE: {mae:.4f}")
        print(f"Pearson Correlation: {pearson_corr:.4f}")
        print(f"Spearman Correlation: {spearman_corr:.4f}")
        
        # Distribution Plot
        plt.figure(figsize=(10, 6))
        sns.histplot(y_true_clean, kde=True, stat="density", element="step", fill=True, label='Ground Truth', color='blue')
        sns.histplot(y_pred_clean, kde=True, stat="density", element="step", fill=True, label='Prediction', color='orange')
        plt.title(f'Purity Distribution Baseline 1\nPearson: {pearson_corr:.4f}, Spearman: {spearman_corr:.4f}')
        plt.xlabel('Purity Value')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.savefig('2-1-1.accuracy-entanglement-baseline-1.pdf')
        plt.close()
        print("Saved 2-1-1.accuracy-entanglement-baseline-1.pdf")
    else:
        print("No valid numeric data found for regression task.")
else:
    print("Columns prediction_2 or output_2 not found.")



--- Task 2: Purity Estimation ---
R2 Score: -2.3371
MAE: 0.2258
Pearson Correlation: -0.2782
Spearman Correlation: -0.3530
Saved 2-1-1.accuracy-entanglement-baseline-1.pdf
